In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
from itertools import product

(c)

In [ ]:
features = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]

signal = pd.read_csv("data/signal_Bs2MuMu.txt", sep=r"\s+", header=None, names=features)
background = pd.read_csv("data/background_combinatorial.txt", sep=r"\s+", header=None, names=features)
signal = signal.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

fisher_df = pd.read_csv("fisher_scores.csv")
top3_features = fisher_df.loc[fisher_df["Feature"] != "MASS", "Feature"].head(3).tolist()
print("top 3 (excluding MASS):", top3_features)

In [ ]:
def get_thresholds(sig_vals, bkg_vals, n_points=15):
    all_vals = np.concatenate([sig_vals, bkg_vals])
    return np.unique(np.percentile(all_vals, np.linspace(2, 98, n_points)))


def refine_thresholds(sig_vals, bkg_vals, best_t, width_fraction=0.08, n_points=25):
    all_vals = np.concatenate([sig_vals, bkg_vals])
    vmin, vmax = all_vals.min(), all_vals.max()
    vrange = vmax - vmin
    low = max(vmin, best_t - width_fraction * vrange)
    high = min(vmax, best_t + width_fraction * vrange)
    return np.unique(np.linspace(low, high, n_points))


def compute_metrics(sig_mask, bkg_mask):
    tp = int(sig_mask.sum())
    fn = int((~sig_mask).sum())
    fp = int(bkg_mask.sum())
    tn = int((~bkg_mask).sum())
    n = len(sig_mask) + len(bkg_mask)
    return dict(TP=tp, FN=fn, FP=fp, TN=tn,
                accuracy=(tp + tn) / n,
                signal_efficiency=tp / (tp + fn),
                background_efficiency=fp / (fp + tn),
                background_rejection=1 - fp / (fp + tn))


def search_cuts(signal_df, background_df, chosen_features, threshold_dict):
    pre_sig, pre_bkg = {}, {}
    for f in chosen_features:
        xs, xb = signal_df[f].to_numpy(), background_df[f].to_numpy()
        ts = threshold_dict[f]
        pre_sig[f] = {">": [xs > t for t in ts], "<": [xs < t for t in ts]}
        pre_bkg[f] = {">": [xb > t for t in ts], "<": [xb < t for t in ts]}

    f1, f2, f3 = chosen_features
    t1s, t2s, t3s = threshold_dict[f1], threshold_dict[f2], threshold_dict[f3]
    best_acc, best_cuts = 0.0, None
    for d1, d2, d3 in product([">", "<"], repeat=3):
        for i in range(len(t1s)):
            s1, b1 = pre_sig[f1][d1][i], pre_bkg[f1][d1][i]
            for j in range(len(t2s)):
                s12 = s1 & pre_sig[f2][d2][j]
                b12 = b1 & pre_bkg[f2][d2][j]
                for k in range(len(t3s)):
                    s123 = s12 & pre_sig[f3][d3][k]
                    b123 = b12 & pre_bkg[f3][d3][k]
                    acc = (s123.sum() + (~b123).sum()) / (len(s123) + len(b123))
                    if acc > best_acc:
                        best_acc = acc
                        best_cuts = {f1: (d1, float(t1s[i])),
                                     f2: (d2, float(t2s[j])),
                                     f3: (d3, float(t3s[k]))}
    return best_cuts, best_acc

In [ ]:
coarse = {f: get_thresholds(signal[f].to_numpy(), background[f].to_numpy()) for f in top3_features}
best_cuts, _ = search_cuts(signal, background, top3_features, coarse)

refined = {
    f: refine_thresholds(signal[f].to_numpy(), background[f].to_numpy(), best_cuts[f][1])
    for f in top3_features
}
best_cuts, acc_final = search_cuts(signal, background, top3_features, refined)


def apply_cuts(df, cut_dict):
    mask = np.ones(len(df), dtype=bool)
    for f, (d, t) in cut_dict.items():
        mask &= (df[f].to_numpy() > t) if d == ">" else (df[f].to_numpy() < t)
    return mask


m = compute_metrics(apply_cuts(signal, best_cuts), apply_cuts(background, best_cuts))
for f, (d, t) in best_cuts.items():
    print(f"  {f} {d} {t:.4f}")
print(f"accuracy = {m['accuracy']:.4f}, sig_eff = {m['signal_efficiency']:.4f}, "
      f"bkg_eff = {m['background_efficiency']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, top3_features):
    all_vals = np.concatenate([signal[f].to_numpy(), background[f].to_numpy()])
    bins = np.linspace(all_vals.min(), all_vals.max(), 60)
    ax.hist(background[f], bins=bins, alpha=0.6, density=True, label="Background")
    ax.hist(signal[f], bins=bins, alpha=0.6, density=True, label="Signal")
    d, thr = best_cuts[f]
    ax.axvline(thr, color="black", linestyle="--", label=f"{d}{thr:.2f}")
    ax.set_title(f); ax.set_xlabel(f); ax.set_ylabel("Density"); ax.legend()

plt.tight_layout()
plt.savefig("plots/rectangular_cuts.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
with open("cut_params.json", "w") as fh:
    json.dump({"cuts": {f: list(v) for f, v in best_cuts.items()}, "accuracy": acc_final}, fh, indent=2)